# HR Analytics – Employee Attrition Analysis

## Project Overview

This notebook reconstructs the analytical work for the **HR Analytics – Employee Attrition** project using the cleaned IBM HR Analytics Employee Attrition dataset.

The objective is to understand employee attrition patterns and identify workforce characteristics associated with higher observed attrition.

**Tools:** Python | Pandas | Matplotlib | PostgreSQL | Power BI


## 1. Business Problem

Employee attrition can create recruitment costs, productivity losses, knowledge gaps, and workforce planning challenges.

This analysis investigates:

- overall employee attrition
- attrition across departments and job roles
- gender and age patterns
- salary and income patterns
- overtime and work-life balance
- job satisfaction
- education field
- employee tenure and experience

The analysis is descriptive: it identifies **observed patterns**, not causal relationships.


## 2. Import Libraries and Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

df = pd.read_csv("HR_Employee_Attrition_Cleaned.csv")
df.head()


## 3. Dataset Overview

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.info()


In [ ]:
df.describe(include="all").T


## 4. Data Quality Check

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing[missing > 0]


In [ ]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate EmployeeNumber:", df["EmployeeNumber"].duplicated().sum())
print("Attrition values:", df["Attrition"].unique())


## 5. Overall Attrition

In [ ]:
total_employees = len(df)
employees_left = (df["Attrition"] == "Yes").sum()
employees_stayed = (df["Attrition"] == "No").sum()
attrition_rate = employees_left / total_employees * 100

summary = pd.DataFrame({
    "Metric": ["Total Employees", "Employees Left", "Employees Stayed", "Attrition Rate (%)"],
    "Value": [total_employees, employees_left, employees_stayed, round(attrition_rate, 2)]
})
summary


### Observation

The dataset contains **1,470 employees**. **237 employees left**, giving an overall observed attrition rate of **16.12%**. The remaining **1,233 employees stayed**.


## 6. Employees by Department

In [ ]:
dept_counts = df["Department"].value_counts().sort_values(ascending=False)
dept_counts


In [ ]:
dept_counts.plot(kind="barh", title="Employees by Department")
plt.xlabel("Employees")
plt.ylabel("Department")
plt.tight_layout()
plt.show()


## 7. Attrition by Department

In [ ]:
dept_attrition = df.groupby("Department").agg(
    total_employees=("EmployeeNumber", "count"),
    employees_left=("Attrition", lambda x: (x == "Yes").sum())
)
dept_attrition["attrition_rate_percent"] = (
    dept_attrition["employees_left"] / dept_attrition["total_employees"] * 100
)
dept_attrition.sort_values("attrition_rate_percent", ascending=False)


### Observation

Sales has the highest observed department attrition rate at approximately **20.63%**, followed by Human Resources at **19.05%**. Research & Development has the lowest of the three departments at approximately **13.84%**.


## 8. Gender Analysis

In [ ]:
gender_analysis = df.groupby("Gender").agg(
    total_employees=("EmployeeNumber", "count"),
    employees_left=("Attrition", lambda x: (x == "Yes").sum())
)
gender_analysis["attrition_rate_percent"] = (
    gender_analysis["employees_left"] / gender_analysis["total_employees"] * 100
)
gender_analysis


### Observation

The observed attrition rate is higher for male employees (**17.01%**) than female employees (**14.80%**). This is a descriptive comparison and should not be interpreted as a causal effect of gender.


## 9. Age Analysis

In [ ]:
bins = [17, 25, 35, 45, 55, 100]
labels = ["18-25", "26-35", "36-45", "46-55", "56+"]
df["AgeGroup"] = pd.cut(df["Age"], bins=bins, labels=labels)

age_analysis = df.groupby("AgeGroup", observed=False).agg(
    total_employees=("EmployeeNumber", "count"),
    employees_left=("Attrition", lambda x: (x == "Yes").sum())
)
age_analysis["attrition_rate_percent"] = (
    age_analysis["employees_left"] / age_analysis["total_employees"] * 100
)
age_analysis


### Observation

The **18–25** group has the highest observed attrition rate at approximately **35.77%**, followed by the **26–35** group at approximately **19.14%**. Attrition is considerably lower in the 36–45 group.


## 10. Salary Analysis

In [ ]:
salary_analysis = df.groupby("Department")["MonthlyIncome"].agg(
    employees="count",
    average_salary="mean",
    minimum_salary="min",
    maximum_salary="max"
).sort_values("average_salary", ascending=False)

salary_analysis.round(2)


In [ ]:
df["SalaryCategory"] = pd.cut(
    df["MonthlyIncome"],
    bins=[-1, 4999, 10000, np.inf],
    labels=["Low", "Medium", "High"]
)

salary_attrition = df.groupby("SalaryCategory", observed=False).agg(
    total_employees=("EmployeeNumber", "count"),
    employees_left=("Attrition", lambda x: (x == "Yes").sum())
)
salary_attrition["attrition_rate_percent"] = (
    salary_attrition["employees_left"] / salary_attrition["total_employees"] * 100
)
salary_attrition


### Observation

The Low salary category has the highest observed attrition rate at approximately **21.76%**, compared with **11.14%** for Medium and **8.90%** for High salary employees.


## 11. Job Role Analysis

In [ ]:
job_role = df.groupby("JobRole").agg(
    total_employees=("EmployeeNumber", "count"),
    employees_left=("Attrition", lambda x: (x == "Yes").sum()),
    average_monthly_income=("MonthlyIncome", "mean")
)
job_role["attrition_rate_percent"] = (
    job_role["employees_left"] / job_role["total_employees"] * 100
)
job_role.sort_values("attrition_rate_percent", ascending=False).round(2)


In [ ]:
role_income = df.groupby("JobRole")["MonthlyIncome"].mean().sort_values(ascending=False)
role_income.plot(kind="barh", title="Average Monthly Income by Job Role")
plt.xlabel("Average Monthly Income")
plt.ylabel("Job Role")
plt.tight_layout()
plt.show()


### Observation

Sales Representatives have the highest observed job-role attrition rate at approximately **39.76%**. Managers and Research Directors have much lower observed attrition rates. Job-role sample sizes should be considered when comparing rates.


## 12. Overtime vs Attrition

In [ ]:
overtime = df.groupby("OverTime").agg(
    total_employees=("EmployeeNumber", "count"),
    employees_left=("Attrition", lambda x: (x == "Yes").sum())
)
overtime["attrition_rate_percent"] = overtime["employees_left"] / overtime["total_employees"] * 100
overtime


### Observation

Employees working overtime show a substantially higher observed attrition rate (**30.53%**) than employees who do not work overtime (**10.44%**). This is one of the clearest patterns in the dataset.


## 13. Job Satisfaction

In [ ]:
job_sat = df.groupby("JobSatisfaction").agg(
    total_employees=("EmployeeNumber", "count"),
    employees_left=("Attrition", lambda x: (x == "Yes").sum())
)
job_sat["attrition_rate_percent"] = job_sat["employees_left"] / job_sat["total_employees"] * 100
job_sat


### Observation

Observed attrition is highest among employees with the lowest job satisfaction score (1), at approximately **22.84%**, and lowest for satisfaction score 4 at approximately **11.33%**.


## 14. Work-Life Balance

In [ ]:
wlb = df.groupby("WorkLifeBalance").agg(
    total_employees=("EmployeeNumber", "count"),
    employees_left=("Attrition", lambda x: (x == "Yes").sum())
)
wlb["attrition_rate_percent"] = wlb["employees_left"] / wlb["total_employees"] * 100
wlb


### Observation

The Work-Life Balance score of 1 has the highest observed attrition rate at **31.25%**. The relationship is not perfectly monotonic across all scores, so this factor should be considered alongside other variables.


## 15. Education Field Analysis

In [ ]:
education = df.groupby("EducationField").agg(
    total_employees=("EmployeeNumber", "count"),
    employees_left=("Attrition", lambda x: (x == "Yes").sum())
)
education["attrition_rate_percent"] = education["employees_left"] / education["total_employees"] * 100
education.sort_values("attrition_rate_percent", ascending=False).round(2)


In [ ]:
education["total_employees"].sort_values(ascending=False).plot(
    kind="bar", title="Employees by Education Field"
)
plt.ylabel("Employees")
plt.xlabel("Education Field")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


### Observation

Human Resources and Technical Degree employees show relatively high observed attrition rates, while Medical and Other have lower observed rates. The Human Resources education-field group is small, so its percentage should be interpreted cautiously.


## 16. Employee Experience and Tenure

In [ ]:
experience = df.groupby("Attrition").agg(
    average_total_working_years=("TotalWorkingYears", "mean"),
    average_years_at_company=("YearsAtCompany", "mean"),
    average_years_current_role=("YearsInCurrentRole", "mean"),
    average_years_with_manager=("YearsWithCurrManager", "mean")
).round(2)

experience


In [ ]:
tenure = df.groupby("Attrition").agg(
    average_years_at_company=("YearsAtCompany", "mean"),
    average_years_current_role=("YearsInCurrentRole", "mean"),
    average_years_since_promotion=("YearsSinceLastPromotion", "mean")
).round(2)
tenure


### Observation

Comparing employees who left with those who stayed provides a useful view of workforce tenure and experience. Shorter company tenure and role tenure among leavers can indicate an area for further retention analysis.


## 17. Key Risk-Factor Summary

In [ ]:
risk_factors = pd.DataFrame({
    "Risk Factor": [
        "Age 18-25",
        "OverTime = Yes",
        "Sales Representative",
        "Work-Life Balance = 1",
        "Low Salary Category"
    ],
    "Observed Attrition Rate (%)": [
        age_analysis.loc["18-25", "attrition_rate_percent"],
        overtime.loc["Yes", "attrition_rate_percent"],
        job_role.loc["Sales Representative", "attrition_rate_percent"],
        wlb.loc[1, "attrition_rate_percent"],
        salary_attrition.loc["Low", "attrition_rate_percent"]
    ]
}).sort_values("Observed Attrition Rate (%)", ascending=False)

risk_factors.round(2)


## 18. Final Business Insights

Based on the descriptive analysis:

1. Overall employee attrition is **16.12%**.
2. Younger employees show higher observed attrition, particularly the 18–25 group.
3. Employees working overtime have substantially higher observed attrition.
4. Sales Representatives show the highest observed attrition rate among job roles.
5. Lower salary categories have higher observed attrition.
6. Low job satisfaction is associated with higher observed attrition.
7. Poor work-life balance is associated with higher observed attrition.
8. Department-level attrition differs across Sales, Human Resources, and Research & Development.

These are **observed associations**, not causal conclusions.


## 19. Business Recommendations

### 1. Focus on Early-Career Retention
Develop stronger onboarding, mentoring, career-development and progression programmes for younger employees.

### 2. Review Overtime Practices
Monitor overtime workload and consider workload balancing, staffing levels and manager interventions.

### 3. Strengthen Retention in High-Risk Roles
Sales Representatives and other higher-attrition roles should receive targeted retention analysis.

### 4. Improve Job Satisfaction
Use regular employee feedback and engagement surveys to identify dissatisfaction drivers.

### 5. Support Work-Life Balance
Review workload, scheduling and flexibility for employees reporting poor work-life balance.

### 6. Review Compensation
Investigate whether lower salary levels are associated with retention challenges and whether pay progression is competitive.



## 20. Power BI Dashboard

The final Power BI dashboard presents:

- Total Employees: **1,470**
- Employees Left: **237**
- Attrition Rate: **16.12%**
- Average Age: **36.92 years**
- Average Monthly Income: approximately **$6.50K**

Main visuals include:

- Employees by Department
- Attrition by Department
- Gender Distribution
- Overtime vs Attrition
- Employee Job Satisfaction
- Employee Age Distribution
- Average Monthly Income by Job Role
- Work-Life Balance
- Employees by Education Field

Interactive filters include Department, Job Role, Education Field and Gender.


## 21. SQL Analysis

The project also contains PostgreSQL scripts covering:

- total employee count
- attrition counts and rate
- department analysis
- gender analysis
- age analysis
- salary analysis
- job-role analysis
- department salary comparisons
- advanced window-function analysis

The SQL scripts are stored separately in the project repository.


## 22. Conclusion

This project demonstrates an end-to-end HR analytics workflow using Python, PostgreSQL and Power BI.

The analysis transforms employee-level data into workforce insights that can support employee-retention and HR decision-making.

The project can be extended with predictive modelling to estimate individual attrition probability and identify employees or segments requiring proactive retention support.
